# Ordinal SVM with TF-IDF

Ordinal-only modeling for `final score` on the 0-4 scale using a stimulus-grouped split, linear SVM, and grid search tuning.

## Setup

The next cell builds the aligned ordinal feature table and the ordered-score target used by the model.

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix, f1_score, mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import silabeador
except ImportError:
    silabeador = None

base = Path.cwd().parent

df = pd.read_excel(base / 'data' / 'combined.xlsx')
idea_units_df = pd.read_excel(base / 'data' / 'idea_units_Spanish_AutoEIT.xlsx')

# Normalize columns and join idea-unit annotations onto the scored responses.
df['final score'] = pd.to_numeric(df['final score'], errors='coerce')
df = df.dropna(subset=['stimulus', 'final transcription', 'final score']).copy()
df['final score'] = df['final score'].astype(int)

idea_units_df.columns = idea_units_df.columns.str.strip().str.lower()
idea_units_df = idea_units_df.rename(columns={'idea units breakdown - established aug 2025': 'idea_units_breakdown'})


def normalize_for_match(text):
    text = unicodedata.normalize('NFKD', str(text).lower())
    text = ''.join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r'[^a-zñ\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


df['stimulus_key'] = df['stimulus'].apply(normalize_for_match)
idea_units_df['stimulus_key'] = idea_units_df['stimulus'].apply(normalize_for_match)
idea_units_df = idea_units_df[['stimulus_key', 'idea_units_breakdown']].drop_duplicates('stimulus_key')
df = df.merge(idea_units_df, on='stimulus_key', how='left')

allowed_scores = {0, 1, 2, 3, 4}
observed_scores = set(df['final score'].unique())
print('rows:', len(df))
print('idea-unit matches:', int(df['idea_units_breakdown'].notna().sum()), '/', len(df))
print('score distribution:')
print(df['final score'].value_counts().sort_index().to_string())
print('only ordinal labels:', observed_scores.issubset(allowed_scores))

if not observed_scores.issubset(allowed_scores):
    raise ValueError(f'unexpected scores found: {sorted(observed_scores - allowed_scores)}')


def tokenize(text):
    return normalize_for_match(text).split()


def normalize_token(token):
    token = token.strip()
    if len(token) > 4 and token.endswith('es'):
        return token[:-2]
    if len(token) > 3 and token.endswith('s'):
        return token[:-1]
    return token


def looks_plural(token):
    token = normalize_token(token)
    return len(token) > 3 and (token.endswith('s') or token.endswith('es'))


def lemmatize_tokens(text):
    return [normalize_token(token) for token in tokenize(text)]


def count_spanish_syllables(text):
    cleaned = normalize_for_match(text)
    if not cleaned:
        return 0

    if silabeador is not None:
        total = 0
        for word in cleaned.split():
            total += len(silabeador.syllabify(word))
        return total

    total = 0
    for word in cleaned.split():
        groups = re.findall(r'[aeiou]+', word)
        total += max(1, len(groups))
    return total


def extract_stimulus_syllables(stimulus_text):
    match = re.search(r'\((\d+)\)\s*$', str(stimulus_text))
    if match:
        return int(match.group(1))
    return count_spanish_syllables(stimulus_text)


def compute_word_alignment(reference_tokens, hypothesis_tokens):
    rows = len(reference_tokens)
    cols = len(hypothesis_tokens)
    costs = np.zeros((rows + 1, cols + 1), dtype=int)
    backtrack = np.zeros((rows + 1, cols + 1), dtype=np.int8)

    for row in range(1, rows + 1):
        costs[row, 0] = row
        backtrack[row, 0] = 1
    for col in range(1, cols + 1):
        costs[0, col] = col
        backtrack[0, col] = 2

    for row in range(1, rows + 1):
        for col in range(1, cols + 1):
            if reference_tokens[row - 1] == hypothesis_tokens[col - 1]:
                substitution_cost = costs[row - 1, col - 1]
            else:
                substitution_cost = costs[row - 1, col - 1] + 1

            deletion_cost = costs[row - 1, col] + 1
            insertion_cost = costs[row, col - 1] + 1
            best_cost = min(substitution_cost, deletion_cost, insertion_cost)
            costs[row, col] = best_cost

            if best_cost == substitution_cost:
                backtrack[row, col] = 0 if reference_tokens[row - 1] == hypothesis_tokens[col - 1] else 3
            elif best_cost == deletion_cost:
                backtrack[row, col] = 1
            else:
                backtrack[row, col] = 2

    hits = substitutions = insertions = deletions = 0
    row = rows
    col = cols
    while row > 0 or col > 0:
        step = backtrack[row, col]
        if step == 0:
            hits += 1
            row -= 1
            col -= 1
        elif step == 3:
            substitutions += 1
            row -= 1
            col -= 1
        elif step == 1:
            deletions += 1
            row -= 1
        else:
            insertions += 1
            col -= 1

    total_reference = max(len(reference_tokens), 1)
    wer = (substitutions + insertions + deletions) / total_reference
    hit_rate = hits / total_reference
    return hits, substitutions, insertions, deletions, wer, hit_rate


NEGATION_WORDS = {'no', 'nunca', 'jamas', 'jamas', 'nadie', 'ningun', 'ninguna', 'ninguno', 'sin', 'ni'}


def calculate_coverage_metrics(transcription, idea_units_breakdown):
    idea_units = []
    if pd.notna(idea_units_breakdown):
        cleaned = re.sub(r'\(\d+\)\s*$', '', str(idea_units_breakdown).strip())
        idea_units = [unit.strip() for unit in cleaned.split('::') if unit.strip()]

    if not idea_units:
        return 0.0, 0, []

    response_tokens = set(lemmatize_tokens(transcription))
    covered_units = []
    for unit in idea_units:
        unit_tokens = [token for token in lemmatize_tokens(unit) if token]
        if unit_tokens and all(token in response_tokens for token in unit_tokens):
            covered_units.append(unit)

    coverage_count = len(covered_units)
    coverage_fraction = coverage_count / len(idea_units)
    return coverage_fraction, coverage_count, covered_units


def build_feature_row(row):
    stimulus = str(row['stimulus'])
    response = str(row['final transcription'])

    stimulus_tokens = lemmatize_tokens(stimulus)
    response_tokens = lemmatize_tokens(response)
    stimulus_token_set = set(stimulus_tokens)
    response_token_set = set(response_tokens)

    hits, substitutions, insertions, deletions, wer, hit_rate = compute_word_alignment(stimulus_tokens, response_tokens)
    coverage_fraction, coverage_count, _ = calculate_coverage_metrics(response, row['idea_units_breakdown'])

    stimulus_syllables = extract_stimulus_syllables(stimulus)
    response_syllables = count_spanish_syllables(response)
    response_length_ratio = len(response_tokens) / max(len(stimulus_tokens), 1)
    syllable_ratio = response_syllables / max(stimulus_syllables, 1)

    prompt_negations = {token for token in stimulus_tokens if token in NEGATION_WORDS}
    response_negations = {token for token in response_tokens if token in NEGATION_WORDS}
    negation_preserved = int(not prompt_negations or bool(prompt_negations & response_negations))

    prompt_plural_tokens = {normalize_token(token) for token in stimulus_tokens if looks_plural(token)}
    response_plural_tokens = {normalize_token(token) for token in response_tokens}
    plural_preserved = int(not prompt_plural_tokens or prompt_plural_tokens.issubset(response_plural_tokens))

    return pd.Series({
        'coverage_fraction': coverage_fraction,
        'coverage_count': coverage_count,
        'wer': wer,
        'substitutions': substitutions,
        'insertions': insertions,
        'deletions': deletions,
        'hit_rate': hit_rate,
        'syllable_ratio': syllable_ratio,
        'response_length_ratio': response_length_ratio,
        'negation_preserved': negation_preserved,
        'plural_preserved': plural_preserved,
    })


feature_frame = df.apply(build_feature_row, axis=1)
df = pd.concat([df, feature_frame], axis=1)

feature_columns = [
    'coverage_fraction',
    'coverage_count',
    'wer',
    'substitutions',
    'insertions',
    'deletions',
    'hit_rate',
    'syllable_ratio',
    'response_length_ratio',
    'negation_preserved',
    'plural_preserved',
]

print('\nfeature columns:')
print(', '.join(feature_columns))
print('\nfeature preview:')
print(df[feature_columns + ['final score']].head().to_string(index=False))

# stimulus-grouped split prevents leakage across repeated prompts
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_val_idx, test_idx = next(splitter.split(df, groups=df['stimulus_key']))
train_val_df = df.iloc[train_val_idx].copy()
test_df = df.iloc[test_idx].copy()

splitter_val = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, val_idx = next(splitter_val.split(train_val_df, groups=train_val_df['stimulus_key']))
train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

X_train = train_df[feature_columns].astype(float)
y_train = train_df['final score']
X_val = val_df[feature_columns].astype(float)
y_val = val_df['final score']
X_test = test_df[feature_columns].astype(float)
y_test = test_df['final score']

print('\nsplit sizes:', len(train_df), len(val_df), len(test_df))
print('stimulus overlap train-val:', len(set(train_df['stimulus_key']) & set(val_df['stimulus_key'])))
print('stimulus overlap train-test:', len(set(train_df['stimulus_key']) & set(test_df['stimulus_key'])))
print('stimulus overlap val-test:', len(set(val_df['stimulus_key']) & set(test_df['stimulus_key'])))

rows: 1964
idea-unit matches: 1964 / 1964
score distribution:
final score
0    338
1    532
2    531
3    268
4    295
only ordinal labels: True

feature columns:
coverage_fraction, coverage_count, wer, substitutions, insertions, deletions, hit_rate, syllable_ratio, response_length_ratio, negation_preserved, plural_preserved

feature preview:
 coverage_fraction  coverage_count  wer  substitutions  insertions  deletions  hit_rate  syllable_ratio  response_length_ratio  negation_preserved  plural_preserved  final score
               0.5             2.0  0.0            0.0         0.0        0.0       1.0        1.142857                    1.0                 1.0               1.0            4
               1.0             4.0  0.0            0.0         0.0        0.0       1.0        1.285714                    1.0                 1.0               1.0            4
               1.0             4.0  0.0            0.0         0.0        0.0       1.0        1.000000                  

## Feature Table

Build the aligned feature matrix and the ordinal 0-4 target used by the threshold ensemble.

In [2]:
class OrdinalThresholdEnsemble:
    def __init__(self, thresholds, C=0.5, random_state=42):
        self.thresholds = thresholds
        self.C = C
        self.random_state = random_state
        self.models = {}

    def fit(self, X, y):
        self.models = {}
        for threshold in self.thresholds:
            model = Pipeline([
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(
                    C=self.C,
                    class_weight='balanced',
                    max_iter=5000,
                    random_state=self.random_state,
                )),
            ])
            model.fit(X, (y >= threshold).astype(int))
            self.models[threshold] = model
        return self

    def predict_threshold_probabilities(self, X):
        probability_columns = [self.models[threshold].predict_proba(X)[:, 1] for threshold in self.thresholds]
        probabilities = np.column_stack(probability_columns)
        probabilities = np.minimum.accumulate(probabilities, axis=1)
        return probabilities

    def predict(self, X):
        return (self.predict_threshold_probabilities(X) >= 0.5).sum(axis=1)


ordinal_model = OrdinalThresholdEnsemble(thresholds=[1, 2, 3, 4], C=0.5, random_state=42)
ordinal_model.fit(X_train, y_train)

train_pred = ordinal_model.predict(X_train)
val_pred = ordinal_model.predict(X_val)
test_pred = ordinal_model.predict(X_test)


def report(name, y_true, y_pred):
    print(f'\n{name}')
    print('accuracy:', round(accuracy_score(y_true, y_pred), 4))
    print('macro_f1:', round(f1_score(y_true, y_pred, average='macro'), 4))
    print('weighted_f1:', round(f1_score(y_true, y_pred, average='weighted'), 4))
    print('mae:', round(mean_absolute_error(y_true, y_pred), 4))
    print('qwk:', round(cohen_kappa_score(y_true, y_pred, weights='quadratic'), 4))


report('ORDINAL TRAIN', y_train, train_pred)
report('ORDINAL VAL', y_val, val_pred)
report('ORDINAL TEST', y_test, test_pred)

print('\nconfusion_matrix_test')
print(confusion_matrix(y_test, test_pred))
print('\nclassification_report_test')
print(classification_report(y_test, test_pred, digits=4))

summary = pd.DataFrame([
    {'split': 'train', 'accuracy': accuracy_score(y_train, train_pred), 'macro_f1': f1_score(y_train, train_pred, average='macro'), 'weighted_f1': f1_score(y_train, train_pred, average='weighted'), 'mae': mean_absolute_error(y_train, train_pred), 'qwk': cohen_kappa_score(y_train, train_pred, weights='quadratic')},
    {'split': 'val', 'accuracy': accuracy_score(y_val, val_pred), 'macro_f1': f1_score(y_val, val_pred, average='macro'), 'weighted_f1': f1_score(y_val, val_pred, average='weighted'), 'mae': mean_absolute_error(y_val, val_pred), 'qwk': cohen_kappa_score(y_val, val_pred, weights='quadratic')},
    {'split': 'test', 'accuracy': accuracy_score(y_test, test_pred), 'macro_f1': f1_score(y_test, test_pred, average='macro'), 'weighted_f1': f1_score(y_test, test_pred, average='weighted'), 'mae': mean_absolute_error(y_test, test_pred), 'qwk': cohen_kappa_score(y_test, test_pred, weights='quadratic')},
])
print('\nsummary')
print(summary.round(4).to_string(index=False))


ORDINAL TRAIN
accuracy: 0.6562
macro_f1: 0.6504
weighted_f1: 0.6429
mae: 0.3682
qwk: 0.8957

ORDINAL VAL
accuracy: 0.6765
macro_f1: 0.6441
weighted_f1: 0.6662
mae: 0.3456
qwk: 0.8906

ORDINAL TEST
accuracy: 0.5537
macro_f1: 0.5534
weighted_f1: 0.5416
mae: 0.4902
qwk: 0.8329

confusion_matrix_test
[[55  2  1  1  0]
 [59 45 13  2  0]
 [ 9 16 64 31  3]
 [ 0  1 12 27 31]
 [ 0  0  0  2 36]]

classification_report_test
              precision    recall  f1-score   support

           0     0.4472    0.9322    0.6044        59
           1     0.7031    0.3782    0.4918       119
           2     0.7111    0.5203    0.6009       123
           3     0.4286    0.3803    0.4030        71
           4     0.5143    0.9474    0.6667        38

    accuracy                         0.5537       410
   macro avg     0.5608    0.6317    0.5534       410
weighted avg     0.6036    0.5537    0.5416       410


summary
split  accuracy  macro_f1  weighted_f1    mae    qwk
train    0.6562    0.6504      

## Evaluation

Print train, validation, and test metrics plus the confusion matrix and summary table.

## Results

This block prints the train/validation/test predictions and the ordinal summary metrics.

In [3]:
test_results = test_df[['stimulus', 'final transcription', 'final score']].copy()
test_results['predicted_score'] = test_pred
test_results['error'] = test_results['predicted_score'] - test_results['final score']
test_results['absolute_error'] = test_results['error'].abs()

print('largest absolute test errors')
print(test_results.sort_values(['absolute_error', 'final score'], ascending=[False, False]).head(12).to_string(index=False))

print('\nper-class support on test')
print(pd.Series(y_test).value_counts().sort_index().to_string())

largest absolute test errors
                                               stimulus                                                   final transcription  final score  predicted_score  error  absolute_error
                           La tarea la tiene Carla. (8) {ah} la no {eh} no oi porque estaba hablando el p- el antes lo siento            0                3      3               3
       A ellos les encantan los libros de romance. (14)                             A ellos le encanten los libros románticas            3                1     -2               2
            Puede que haga mucho calor esta noche. (12)                                    Puedo hace mucha calor este manaña            2                0     -2               2
 El trabajo no fue tan fácil como me habían dicho. (17)                       El tra- el trabajo no . es fácil pero han dicho            2                0     -2               2
                ¿Qué dice usted que va a hacer hoy? (9)                     